# Car Racing — PPO with PyTorch

**Environment:** `CarRacing-v3` (top-down, continuous)  
**Algorithm:** Proximal Policy Optimization (PPO-Clip) with CNN actor-critic  

| Property | Detail |
|----------|--------|
| Observation | `(96, 96, 3)` RGB pixel frame |
| Action | `(3,)` continuous — steering · gas · brake |
| Reward | +1000/N tiles visited − 0.1/frame |

## 1. Imports

In [5]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

print(f"Gymnasium : {gym.__version__}")
print(f"PyTorch   : {torch.__version__}")
DEVICE = torch.device("mps" if torch.backends.mps.is_available()
                       else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Device    : {DEVICE}")

Gymnasium : 1.1.1
PyTorch   : 2.12.0
Device    : mps


## 2. CNN Actor & Critic Networks

The observation is a raw **96×96 RGB** image, so both networks share a CNN backbone.

| Layer | Output shape | Notes |
|-------|-------------|-------|
| Conv(3→32, 8×8, s=4) | 23×23×32 | |
| Conv(32→64, 4×4, s=2) | 10×10×64 | |
| Conv(64→64, 3×3, s=1) | 8×8×64 | |
| Flatten → FC 512 | 512 | shared feature |
| Actor head | μ ∈ (3,), log σ ∈ (3,) | Gaussian policy |
| Critic head | scalar V(s) | |

In [6]:
N_ACTIONS  = 3
HIDDEN     = 512
LOG_STD_MIN, LOG_STD_MAX = -5, 2


class CNNBase(nn.Module):
    """Shared CNN feature extractor for 96×96 RGB observations."""

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=8, stride=4),   # → (23,23)
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),  # → (10,10)
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),  # → (8,8)
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, HIDDEN),
            nn.ReLU(),
        )
        self.out_dim = HIDDEN

    def forward(self, obs):
        """obs: (B,H,W,C) uint8 or (H,W,C) → normalised feature vector."""
        if obs.dim() == 3:
            obs = obs.unsqueeze(0)
        x = obs.permute(0, 3, 1, 2).float() / 255.0
        return self.net(x)


class ContinuousActor(nn.Module):
    """Gaussian policy: mean (tanh-bounded) + learned log_std per action."""

    def __init__(self, n_actions=N_ACTIONS, lr=3e-4):
        super().__init__()
        self.backbone     = CNNBase()
        self.mean_head    = nn.Linear(HIDDEN, n_actions)
        self.log_std_head = nn.Linear(HIDDEN, n_actions)
        self.optimizer    = optim.Adam(self.parameters(), lr=lr)

    def forward(self, obs):
        feat    = self.backbone(obs)
        mean    = torch.tanh(self.mean_head(feat))
        log_std = self.log_std_head(feat).clamp(LOG_STD_MIN, LOG_STD_MAX)
        return mean, log_std

    @torch.no_grad()
    def select_action(self, obs_np):
        """Sample action and return (action_np, detached log_prob)."""
        obs_t = torch.tensor(obs_np, dtype=torch.uint8).to(DEVICE)
        mean, log_std = self.forward(obs_t)
        std  = log_std.exp()
        dist = torch.distributions.Normal(mean, std)
        raw  = dist.sample()
        act  = torch.tanh(raw)
        # Tanh-squashing log-prob correction
        log_prob = (dist.log_prob(raw) - torch.log(1 - act.pow(2) + 1e-6)).sum(-1)
        return act.squeeze(0).cpu().numpy(), log_prob.detach()

    def evaluate(self, obs_t, actions_t):
        mean, log_std = self.forward(obs_t)
        std  = log_std.exp()
        dist = torch.distributions.Normal(mean, std)
        raw  = torch.atanh(actions_t.clamp(-1 + 1e-6, 1 - 1e-6))
        log_prob = (dist.log_prob(raw) - torch.log(1 - actions_t.pow(2) + 1e-6)).sum(-1)
        entropy  = dist.entropy().sum(-1)
        return log_prob, entropy

    def learn(self, obs_t, actions_t, old_log_probs, advantages,
              epsilon=0.2, entropy_coef=0.01):
        new_log_probs, entropy = self.evaluate(obs_t, actions_t)
        ratio  = torch.exp(new_log_probs - old_log_probs)
        surr1  = ratio * advantages
        surr2  = torch.clamp(ratio, 1 - epsilon, 1 + epsilon) * advantages
        loss   = -torch.min(surr1, surr2).mean() - entropy_coef * entropy.mean()
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.parameters(), 0.5)
        self.optimizer.step()
        return loss.item()


class ContinuousCritic(nn.Module):
    """Value network V(s) for pixel observations."""

    def __init__(self, lr=3e-4):
        super().__init__()
        self.backbone   = CNNBase()
        self.value_head = nn.Linear(HIDDEN, 1)
        self.optimizer  = optim.Adam(self.parameters(), lr=lr)

    def forward(self, obs):
        return self.value_head(self.backbone(obs)).squeeze(-1)

    @torch.no_grad()
    def value(self, obs_np):
        obs_t = torch.tensor(obs_np, dtype=torch.uint8).to(DEVICE)
        return self.forward(obs_t).item()

    def learn(self, obs_t, returns):
        loss = F.mse_loss(self.forward(obs_t), returns)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.parameters(), 0.5)
        self.optimizer.step()
        return loss.item()


print("ContinuousActor and ContinuousCritic defined.")

ContinuousActor and ContinuousCritic defined.


## 3. PPO Agent

Same PPO-Clip loop as FrozenLake but adapted for:
- **Continuous actions** (Gaussian policy, tanh squashing)  
- **Pixel observations** stored in the rollout buffer  
- **GAE** (Generalised Advantage Estimation) for lower-variance advantages

In [7]:
class CarPPOAgent:
    """
    PPO-Clip agent for continuous-action pixel environments.

    Parameters
    ----------
    lr             : Adam learning rate
    gamma          : discount factor
    gae_lambda     : GAE smoothing (0 = TD, 1 = MC)
    epsilon        : PPO clip range
    entropy_coef   : entropy bonus weight
    k_epochs       : gradient updates per collected batch
    rollout_length : environment steps per epoch
    """

    def __init__(self, lr=3e-4, gamma=0.99, gae_lambda=0.95,
                 epsilon=0.2, entropy_coef=0.01, k_epochs=4,
                 rollout_length=512):
        self.actor    = ContinuousActor(N_ACTIONS, lr).to(DEVICE)
        self.critic   = ContinuousCritic(lr).to(DEVICE)
        self.gamma    = gamma
        self.lam      = gae_lambda
        self.epsilon  = epsilon
        self.ent_coef = entropy_coef
        self.k_epochs = k_epochs
        self.rollout_length = rollout_length

    # ------------------------------------------------------------------

    def _collect(self, env, obs):
        """Gather rollout_length transitions. Returns buffer and final obs."""
        buf = []
        for _ in range(self.rollout_length):
            action, log_prob = self.actor.select_action(obs)
            value            = self.critic.value(obs)
            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            buf.append((obs, action, float(reward), done, log_prob, value))
            obs = env.reset()[0] if done else next_obs
        return buf, obs

    def _gae(self, rewards, dones, values):
        """Generalised Advantage Estimation."""
        advantages, gae = [], 0.0
        for r, d, v in zip(reversed(rewards), reversed(dones), reversed(values)):
            delta = r + self.gamma * v * (1 - d) - v
            gae   = delta + self.gamma * self.lam * (1 - d) * gae
            advantages.insert(0, gae)
        returns = [a + v for a, v in zip(advantages, values)]
        return (torch.tensor(advantages, dtype=torch.float32),
                torch.tensor(returns,    dtype=torch.float32))

    # ------------------------------------------------------------------

    def train(self, env, n_updates=500, seed=42):
        """Run the training loop. Returns history dict."""
        obs, _ = env.reset(seed=seed)
        history = {"actor_loss": [], "critic_loss": [], "ep_return": []}
        ep_return, ep_returns = 0.0, []

        for update in range(1, n_updates + 1):
            buf, obs = self._collect(env, obs)

            obss      = np.array([b[0] for b in buf], dtype=np.uint8)
            actions   = np.array([b[1] for b in buf], dtype=np.float32)
            rewards   = [b[2] for b in buf]
            dones     = [b[3] for b in buf]
            log_probs = torch.stack([b[4] for b in buf]).to(DEVICE)
            values    = [b[5] for b in buf]

            # Episode returns for logging
            for r, d in zip(rewards, dones):
                ep_return += r
                if d:
                    ep_returns.append(ep_return)
                    ep_return = 0.0
            history["ep_return"].append(
                np.mean(ep_returns[-10:]) if ep_returns else 0.0)

            advantages, returns = self._gae(rewards, dones, values)
            advantages = ((advantages - advantages.mean())
                          / (advantages.std() + 1e-8)).to(DEVICE)
            returns = returns.to(DEVICE)

            obs_t  = torch.tensor(obss,    dtype=torch.uint8).to(DEVICE)
            act_t  = torch.tensor(actions, dtype=torch.float32).to(DEVICE)

            a_losses, c_losses = [], []
            for _ in range(self.k_epochs):
                a_losses.append(
                    self.actor.learn(obs_t, act_t, log_probs, advantages,
                                     self.epsilon, self.ent_coef))
                c_losses.append(self.critic.learn(obs_t, returns))

            history["actor_loss"].append(sum(a_losses)  / self.k_epochs)
            history["critic_loss"].append(sum(c_losses) / self.k_epochs)

            if update % 50 == 0:
                print(f"Update {update:4d}/{n_updates}"
                      f"  ep_ret={history['ep_return'][-1]:7.1f}"
                      f"  a_loss={history['actor_loss'][-1]:8.4f}"
                      f"  c_loss={history['critic_loss'][-1]:8.4f}")

        env.close()
        return history

    @staticmethod
    def plot(history):
        fig, axes = plt.subplots(1, 3, figsize=(16, 4))
        fig.suptitle("PPO Training — CarRacing-v3", fontsize=14, fontweight="bold")
        panels = [
            (axes[0], "ep_return",   "Episode Return (10-ep avg)", "Return",     "#2ca02c"),
            (axes[1], "critic_loss", "Critic Loss (MSE)",          "Loss",       "#d62728"),
            (axes[2], "actor_loss",  "Actor Loss (PPO-Clip)",      "Loss",       "#1f77b4"),
        ]
        for ax, key, title, ylabel, color in panels:
            ax.plot(history[key], color=color, linewidth=2)
            ax.set_title(title, fontsize=11, fontweight="bold")
            ax.set_xlabel("Update Step")
            ax.set_ylabel(ylabel)
            ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()


print("CarPPOAgent defined.")

CarPPOAgent defined.


## 4. Train

| Hyperparameter | Value |
|----------------|-------|
| `rollout_length` | 512 |
| `k_epochs` | 4 |
| `lr` | 3e-4 |
| `gamma` | 0.99 |
| `gae_lambda` | 0.95 |
| `epsilon` | 0.2 |
| `entropy_coef` | 0.01 |

In [ ]:
import pygame
import numpy as np


class CarLiveRenderer:
    """
    Split pygame window for CarRacing training:
      - Left  : CarRacing game frame (rgb_array, auto-sized)
      - Right : stats panel — epoch, buffer fill, then sparkline charts for
                Ep Return · Critic Loss · Actor Loss · Avg Steer · Avg Gas · Avg Brake
    """

    PANEL_W   = 320
    BG_COLOR  = (18, 18, 28)
    DIV_CLR   = (55, 55, 80)
    CHART_H   = 34    # height of each sparkline
    ACTION_H  = 28    # height of action sparklines (slightly smaller)

    def __init__(self, game_w: int = 600, game_h: int = 400):
        pygame.init()
        self.game_w = game_w
        self.game_h = game_h
        self._screen_ready = False
        self.font_xl = pygame.font.SysFont("monospace", 26, bold=True)
        self.font_md = pygame.font.SysFont("monospace", 19, bold=True)
        self.font_sm = pygame.font.SysFont("monospace", 12)

    def _ensure_screen(self, frame_h, frame_w):
        if not self._screen_ready:
            self.game_h = max(frame_h, 640)   # tall enough for all charts
            self.game_w = frame_w
            self.screen = pygame.display.set_mode(
                (self.game_w + self.PANEL_W, self.game_h))
            pygame.display.set_caption("CarRacing — PPO Live Training")
            self._screen_ready = True

    # ------------------------------------------------------------------

    def draw(self, frame_rgb, update, n_updates, history: dict,
             buf_step: int = 0, rollout_length: int = 0) -> bool:
        """Render game + stats panel. Returns False if user closes window."""
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                return False

        h, w = frame_rgb.shape[:2]
        self._ensure_screen(h, w)

        # ── Left pane: game frame ─────────────────────────────────────
        surf = pygame.surfarray.make_surface(np.transpose(frame_rgb, (1, 0, 2)))
        surf = pygame.transform.scale(surf, (self.game_w, self.game_h))
        self.screen.blit(surf, (0, 0))

        # ── Right pane ────────────────────────────────────────────────
        self.screen.fill(self.BG_COLOR,
                         pygame.Rect(self.game_w, 0, self.PANEL_W, self.game_h))
        pygame.draw.line(self.screen, self.DIV_CLR,
                         (self.game_w, 0), (self.game_w, self.game_h), 3)

        x     = self.game_w + 16
        y     = 16
        bar_w = self.PANEL_W - 32

        # ── Header ────────────────────────────────────────────────────
        self.screen.blit(
            self.font_xl.render("PPO Stats", True, (210, 210, 255)), (x, y))
        y += 36

        # ── Epoch ─────────────────────────────────────────────────────
        epoch_col = (110, 230, 110)
        self._lbl(x, y, "Epoch");  y += 15
        self.screen.blit(
            self.font_md.render(f"{update} / {n_updates}", True, epoch_col), (x, y))
        y += 24
        pygame.draw.rect(self.screen, (45, 45, 65), (x, y, bar_w, 10), border_radius=5)
        fill = max(4, int(bar_w * update / n_updates))
        pygame.draw.rect(self.screen, epoch_col, (x, y, fill, 10), border_radius=5)
        pct = self.font_sm.render(f"{100 * update // n_updates}%", True, (155, 155, 180))
        self.screen.blit(pct, (x + bar_w + 5, y))
        y += 20

        # ── Buffer fill ───────────────────────────────────────────────
        if rollout_length > 0:
            buf_col = (130, 180, 255)
            self._lbl(x, y, f"Buffer  {buf_step} / {rollout_length}");  y += 15
            pygame.draw.rect(self.screen, (45, 45, 65), (x, y, bar_w, 9), border_radius=4)
            buf_fill = max(2, int(bar_w * buf_step / rollout_length))
            pygame.draw.rect(self.screen, buf_col, (x, y, buf_fill, 9), border_radius=4)
            y += 16

        # Divider
        pygame.draw.line(self.screen, self.DIV_CLR, (x, y + 4), (x + bar_w, y + 4), 1)
        y += 12

        # ── Training metric charts ─────────────────────────────────────
        train_charts = [
            ("Ep Return",   history.get("ep_return",   []), (90,  210, 100), None),
            ("Critic Loss", history.get("critic_loss", []), (255, 90,  90),  None),
            ("Actor Loss",  history.get("actor_loss",  []), (255, 185, 70),  None),
        ]
        for label, values, color, fr in train_charts:
            y = self._draw_chart(x, y, bar_w, self.CHART_H, values, color, label, fr)
            y += 6

        # Divider
        pygame.draw.line(self.screen, self.DIV_CLR, (x, y + 2), (x + bar_w, y + 2), 1)
        y += 10

        # ── Action history charts ─────────────────────────────────────
        action_charts = [
            ("Avg Steer",  history.get("mean_steer", []), (255, 220, 70),  (-1.0, 1.0)),
            ("Avg Gas",    history.get("mean_gas",   []), (100, 220, 100), ( 0.0, 1.0)),
            ("Avg Brake",  history.get("mean_brake", []), (255, 90,  90),  ( 0.0, 1.0)),
        ]
        for label, values, color, fr in action_charts:
            y = self._draw_chart(x, y, bar_w, self.ACTION_H, values, color, label, fr)
            y += 6

        # Divider + current values
        pygame.draw.line(self.screen, self.DIV_CLR, (x, y + 2), (x + bar_w, y + 2), 1)
        y += 10

        def _row(label, value, color, fmt):
            nonlocal y
            self.screen.blit(
                self.font_sm.render(f"{label}: {fmt.format(value)}", True, color), (x, y))
            y += 17

        ep = history["ep_return"][-1]   if history["ep_return"]   else 0.0
        al = history["actor_loss"][-1]  if history["actor_loss"]  else 0.0
        cl = history["critic_loss"][-1] if history["critic_loss"] else 0.0
        st = history["mean_steer"][-1]  if history["mean_steer"]  else 0.0
        ga = history["mean_gas"][-1]    if history["mean_gas"]    else 0.0
        br = history["mean_brake"][-1]  if history["mean_brake"]  else 0.0

        _row("Ep Return",  ep, (90, 200, 255) if ep >= 500 else (255, 155, 70), "{:.1f}")
        _row("Actor Loss", al, (255, 185,  70), "{:.4f}")
        _row("Crit Loss",  cl, (255,  90,  90), "{:.4f}")
        _row("Steer",      st, (255, 220,  70), "{:+.3f}")
        _row("Gas",        ga, (100, 220, 100), "{:.3f}")
        _row("Brake",      br, (255,  90,  90), "{:.3f}")

        pygame.display.flip()
        return True

    # ------------------------------------------------------------------

    def _draw_chart(self, x, y, w, h, values, color, label, fixed_range=None):
        self._lbl(x, y, label)
        if values:
            cur = values[-1]
            fmt = f"{cur:+.3f}" if fixed_range == (-1.0, 1.0) else f"{cur:.3f}"
            cur_surf = self.font_sm.render(fmt, True, color)
            self.screen.blit(cur_surf, (x + w - cur_surf.get_width(), y))
        y += 14
        pygame.draw.rect(self.screen, (32, 32, 50), (x, y, w, h), border_radius=3)
        if len(values) >= 2:
            mn, mx = (fixed_range if fixed_range else (min(values), max(values)))
            rng = max(mx - mn, 1e-8)
            n   = len(values)
            pts = [(x + int(i * (w - 1) / max(n - 1, 1)),
                    y + h - 2 - int((v - mn) / rng * (h - 4)))
                   for i, v in enumerate(values)]
            fill_surf = pygame.Surface((w, h), pygame.SRCALPHA)
            pygame.draw.polygon(fill_surf, (*color, 35),
                                [(p[0] - x, p[1] - y) for p in
                                 [(x, y + h), *pts, (x + w - 1, y + h)]])
            self.screen.blit(fill_surf, (x, y))
            pygame.draw.lines(self.screen, color, False, pts, 2)
            # zero-line for steer chart
            if fixed_range == (-1.0, 1.0):
                zy = y + h - 2 - int((0.0 - mn) / rng * (h - 4))
                pygame.draw.line(self.screen, (70, 70, 90), (x, zy), (x + w, zy), 1)
        return y + h

    def _lbl(self, x, y, text):
        self.screen.blit(self.font_sm.render(text, True, (120, 125, 160)), (x, y))

    def close(self):
        pygame.quit()


# ──────────────────────────────────────────────────────────────────────

def train_live_car(agent: CarPPOAgent, env, n_updates: int = 500, seed: int = 42,
                   step_delay_ms: int = 0, slow_after: int = 400,
                   slow_delay_ms: int = 30) -> dict:
    """
    CarRacing PPO training loop with live split pygame window.

    Parameters
    ----------
    agent          : CarPPOAgent instance
    env            : CarRacing env with render_mode="rgb_array"
    n_updates      : total training epochs
    seed           : random seed
    step_delay_ms  : per-step delay in ms before slow_after (0 = full speed)
    slow_after     : epoch after which slow_delay_ms is applied
    slow_delay_ms  : per-step delay in ms once slow mode kicks in
    """
    renderer           = CarLiveRenderer()
    obs, _             = env.reset(seed=seed)
    history            = {"actor_loss": [], "critic_loss": [], "ep_return": [],
                          "mean_steer": [], "mean_gas":   [], "mean_brake": []}
    ep_return, ep_rets = 0.0, []
    running            = True

    for update in range(1, n_updates + 1):
        if not running:
            break

        effective_delay = slow_delay_ms if update > slow_after else step_delay_ms

        # ── 1. Collect rollout ────────────────────────────────────────
        buf = []
        for buf_step in range(agent.rollout_length):
            action, log_prob = agent.actor.select_action(obs)
            value            = agent.critic.value(obs)
            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            buf.append((obs, action, float(reward), done, log_prob, value))
            ep_return += reward
            if done:
                ep_rets.append(ep_return);  ep_return = 0.0
            obs = env.reset()[0] if done else next_obs

            frame = env.render()
            if not renderer.draw(frame, update, n_updates, history,
                                 buf_step=buf_step + 1,
                                 rollout_length=agent.rollout_length):
                running = False
                break
            if effective_delay:
                pygame.time.delay(effective_delay)

        # ── 2. Unpack ─────────────────────────────────────────────────
        obss      = np.array([b[0] for b in buf], dtype=np.uint8)
        actions   = np.array([b[1] for b in buf], dtype=np.float32)
        rewards   = [b[2] for b in buf]
        dones     = [b[3] for b in buf]
        log_probs = torch.stack([b[4] for b in buf]).to(DEVICE)
        values    = [b[5] for b in buf]

        history["ep_return"].append(np.mean(ep_rets[-10:]) if ep_rets else 0.0)
        # Per-epoch mean action values
        history["mean_steer"].append(float(np.mean(actions[:, 0])))
        history["mean_gas"].append(  float(np.mean(actions[:, 1])))
        history["mean_brake"].append(float(np.mean(actions[:, 2])))

        # ── 3. GAE ────────────────────────────────────────────────────
        advantages, returns = agent._gae(rewards, dones, values)
        advantages = ((advantages - advantages.mean())
                      / (advantages.std() + 1e-8)).to(DEVICE)
        returns = returns.to(DEVICE)

        obs_t = torch.tensor(obss,    dtype=torch.uint8).to(DEVICE)
        act_t = torch.tensor(actions, dtype=torch.float32).to(DEVICE)

        # ── 4. PPO updates ────────────────────────────────────────────
        a_losses, c_losses = [], []
        for _ in range(agent.k_epochs):
            a_losses.append(
                agent.actor.learn(obs_t, act_t, log_probs, advantages,
                                  agent.epsilon, agent.ent_coef))
            c_losses.append(agent.critic.learn(obs_t, returns))

        history["actor_loss"].append(sum(a_losses)  / agent.k_epochs)
        history["critic_loss"].append(sum(c_losses) / agent.k_epochs)

        if update % 50 == 0:
            print(f"Update {update:4d}/{n_updates}"
                  f"  ep_ret={history['ep_return'][-1]:7.1f}"
                  f"  steer={history['mean_steer'][-1]:+.3f}"
                  f"  gas={history['mean_gas'][-1]:.3f}"
                  f"  brake={history['mean_brake'][-1]:.3f}"
                  + ("  [slow]" if update > slow_after else ""))

    env.close()
    renderer.close()
    return history


print("CarLiveRenderer and train_live_car defined.")

CarLiveRenderer and train_live_car defined.


/Users/juansebastianvargastorres/UTS/Study/frozen-lake/.venv/lib/python3.14/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [9]:
# env with rgb_array so we drive the pygame window ourselves
env_live = gym.make("CarRacing-v3", render_mode="rgb_array", continuous=True)

agent_live = CarPPOAgent(
    lr=3e-4, gamma=0.99, gae_lambda=0.95,
    epsilon=0.2, entropy_coef=0.01,
    k_epochs=5, rollout_length=512,
)

# Epochs 1-400 → full speed
# Epochs 401-500 → 30 ms/step so you can watch the car drive
history_live = train_live_car(
    agent_live, env_live,
    n_updates=500, seed=42,
    step_delay_ms=0,
    slow_after=400,
    slow_delay_ms=30,
)

CarPPOAgent.plot(history_live)

RuntimeError: Mismatched Tensor types in NNPack convolutionOutput

In [9]:
import gymnasium as gym

# 1. Initialize the environment
# continuous=True means actions are arrays [steering, gas, brake]
env = gym.make("CarRacing-v3", render_mode=None, continuous=True)

# 2. Reset the environment to start a new episode
# In modern Gymnasium, reset() returns a tuple: (observation, info)
observation, info = env.reset()

print("Environment started. Running 500 steps with random actions...")

# 3. Run the simulation loop
for step in range(500):
    # Sample a random action from the continuous action space
    # action = [steering (-1 to 1), gas (0 to 1), brake (0 to 1)]
    action = env.action_space.sample()
    
    # Pass the action to the environment
    observation, reward, terminated, truncated, info = env.step(action)
    
    
    # Optional: Print out the reward every 100 steps to monitor progress
    if step % 100 == 0:
        print(f"Step {step}: Action taken = {action}, Reward = {reward:.2f} terminated = {terminated} truncated = {truncated}")

    # 4. Check if the episode is over
    # terminated = reached the goal or failed completely
    # truncated  = ran out of time/max steps
    if terminated or truncated:
        print("Episode ended! Resetting environment...")
        observation, info = env.reset()

# 5. Clean up and close the environment
env.close()
print("Environment closed.")

Environment started. Running 500 steps with random actions...
Step 0: Action taken = [-0.59238124  0.36206278  0.07013902], Reward = 7.81 terminated = False truncated = False
Step 100: Action taken = [-0.90343    0.9266529  0.9296079], Reward = -0.10 terminated = False truncated = False
Step 200: Action taken = [0.8112138  0.5013063  0.69146746], Reward = -0.10 terminated = False truncated = False
Step 300: Action taken = [0.3733023  0.9333083  0.48778042], Reward = -0.10 terminated = False truncated = False
Step 400: Action taken = [0.9043338  0.2662108  0.66066974], Reward = -0.10 terminated = False truncated = False
Environment closed.
